## Futásidő-összevetés: rprev referencia (naiv K-szoros) vs. kiterjesztett több indexdátumos implementáció

Ez a notebook a szakdolgozat 5. fejezetében rögzített több indexdátumos kiterjesztéshez kapcsolódó **futásidő-mérési protokollt** valósítja meg. A mérés célja annak kvantifikálása, hogy azonos bemeneti adaton és azonos szimulációs beállítások mellett milyen számítási igénye van  
(i) a referencia-megoldásnak, ahol a $\{t_k\}_{k=1}^K$ rácsot **$K$ külön `prevalence()` hívással** állítjuk elő, illetve  
(ii) a kiterjesztett rprev-implementációnak, ahol az `index` argumentum több időpontot ad meg, és a becslés **egyetlen eljárásfuttatásban** állítja elő a teljes prevalenciavektort és annak összefoglalóját.

### Kapcsolódás a dolgozathoz (5. fejezet)

A kiterjesztés matematikai tartalma röviden: adott $t_1 < \cdots < t_K$ rácson, az $r$-edik Monte Carlo futásban a futáson belül előállított incidens esetek $(D_{i\ell}^{(r)}, \mathbf{x}_{i\ell}^{(r)})$ és a futáson belül rögzített túlélési függvény $S^{(r)}(\cdot \mid \mathbf{x})$ alapján minden $t_k$ időpontra képezzük a prevalens státuszt, majd időpontonként összegezve kapjuk a $P^{(r)}(t_k)$ prevalenciakomponenseket és a futások feletti összefoglaló statisztikákat.

Csomagszinten a notebook azt a működést tekinti „kiterjesztett” megvalósításnak, ahol a `prevalence()` az `index` bemenetet rendezett `index_dates = (t_1,\ldots,t_K)` vektorrá normalizálja, a belső szimulációs réteg futásonként az `index_dates` rácson szolgáltat státuszinformációt, amelyből az összegző réteg időpontonként képez prevalencia-összefoglalót. A kimeneti objektum tartalmazza az `index_dates` vektort és az időpontindexelt eredményeket; a `print()`/`summary()` metódusok ezek szerint rendezve jelenítenek meg.

### Mérési egységek és protokoll

A dolgozat terminológiáját követve:

- **Alapkiértékelés:** a `prevalence()` egy futtatása rögzített $t$ mellett.
- **Eljárásfuttatás:** a $\{t_k\}_{k=1}^K$ időpontrács teljes kimenetének előállítása.

A referencia-megoldásban egy eljárásfuttatás **$K$ darab alapkiértékelésből** áll (külön-külön időpontra), míg a kiterjesztett megoldásban ugyanez **egyetlen alapkiértékelésként** valósul meg (általános esetben $K \ge 1$, speciális esetben $K=1$).

A notebook minden paraméterkombinációra ismételt futásidő-mérést végez (`reps`), és az összidőt `system.time(...)[["elapsed"]]` alapján rögzíti. A riportolt érték minden beállításpontra az ismétlések átlaga.

### Bemeneti adat és indexdátumok

A méréshez az rprev csomag `prevsim` példaadata szolgál alapul; ebből visszatevéses mintavétellel áll elő az $N$ soros, regiszterjellegű bemeneti tábla (a mérésben az $N$ a „rekordszámot” jelöli). Az indexdátumok egy rögzített kezdődátumtól (`index_start`) fix naplépéssel (`index_step_days`) generált, rendezett $K$ hosszú sorozatot alkotnak.

A notebookban az indexdátumok generálása determinisztikus: adott $K$, `index_start` és `index_step_days` mellett az `index_dates` teljesen reprodukálható, és a különböző mérési pontok között kizárólag a konfigurációs paraméterek változnak.

### Szimulációs és bootstrap beállítások

A mérések rácsa a következő fő paraméterek mentén szervezett:

- $N$: adatméret (rekordszám),
- $K$: indexdátumok száma,
- `N_boot`: bootstrap beállítás (túlélési görbék kezelése),
- opcionális `iterations`: csak akkor kerül átadásra, ha az aktuális `prevalence()` szignatúrája támogatja.

A notebook a `call_prevalence_safe()` segédfüggvénnyel gondoskodik arról, hogy a `prevalence()` csak a ténylegesen létező argumentumokat kapja meg. Ennek szerepe az, hogy a notebook csomagverziók között is futtatható maradjon, miközben az összehasonlítás logikája változatlan.

### Numerikus megjegyzés a több indexdátumos kiértékeléshez

A több indexdátumos kiértékelésben központi mennyiség az esetszintű diagnózisidő és az indexidőpont különbsége. Jelölje

$$
\Delta_{i\ell k} = t_k - D_{i\ell}.
$$

A $\Delta_{i\ell k} < 0$ esetek diagnózis előtti időpontot jelentenek, ahol prevalens státusz nem értelmezhető; ezt egy maszkkal kezeljük, például

$$
M_{i\ell k} = \mathbb{I}\{\Delta_{i\ell k} \ge 0\}.
$$

A kiértékelés ezek után a maszk szerinti pontokon végzi el a túlélési valószínűségek és a státuszindikátorok képzését, majd időpontonkénti aggregálással adja a prevalencia-idősor összetevőit.

### Kimenet

A futás végén egy eredménytábla készül, amely minden paraméterpontra tartalmazza:

- `N`, `K`, `N_boot`,
- `reps`,
- `iterations` (ha értelmezett),
- `elapsed_sec_mean`: az eljárásfuttatás átlagos falióra-ideje másodpercben.

A táblázat opcionálisan CSV-be is menthető, így a dolgozatban szereplő futásidő-ábrák és összegzések közvetlenül reprodukálhatók.

### Megjegyzés a dátumkezelésről

Az indexdátumok a hívás előtt egységesen `YYYY-MM-DD` formátumú karakterláncokká kerülnek alakításra. Ez a megoldás elkerüli azt a gyakorlati problémát, hogy ciklusváltozóként iterálva a `Date` osztályattribútum elveszhet, ami `NA`-vá parsolt indexdátumhoz és hibás futáshoz vezetne.

In [4]:
# =========================
# rprev pilot futásidő-mérés (baseline: K-szor prevalence())
# Kimenet: N, K, N_boot, reps, iterations, elapsed_sec_mean
# =========================

suppressPackageStartupMessages({
  library(rprev)
  library(survival)
})

# ---- KÖZPONTI BEÁLLÍTÁSOK ----
CFG <- list(
  N_values              = c(1e3, 1e4),  # adatméret (rekordszám)
  K_values              = c(1, 10, 30),  # indexdátumok száma (K pont)
  N_boot_values         = c(100, 500, 1000),        # bootstrap beállítás
  iterations_values     = c(1000),           # ha prevalence() nem ismeri, safe wrapper kiszűri
  reps                  = 3L,                # időmérés ismétlésszáma
  num_years_to_estimate = c(15),
  index_start           = as.Date("2013-01-30"),
  index_step_days       = 30L
)

# ---- Alapadat ----
data(prevsim, package = "rprev")
base_df <- as.data.frame(prevsim)

# ---- Segédfüggvény: N soros "registry" a prevsim-ből ----
make_registry <- function(N, seed = 1L) {
  set.seed(as.integer(seed))
  base_df[sample.int(nrow(base_df), size = as.integer(N), replace = TRUE), , drop = FALSE]
}

# ---- Segédfüggvény: K db indexdátum (Date) ----
make_index_dates <- function(K, start_date, step_days) {
  seq.Date(from = start_date, by = as.integer(step_days), length.out = as.integer(K))
}

# ---- Biztonságos prevalence-hívás: csak a létező argumentumokat adja át ----
call_prevalence_safe <- function(...) {
  args <- list(...)
  fmls <- names(formals(rprev::prevalence))
  args <- args[names(args) %in% fmls]
  do.call(rprev::prevalence, args)
}

# ---- Egy futás: 1 indexdátumra ----
# index_date_chr: "YYYY-MM-DD"
run_rprev_once <- function(dat, index_date_chr, N_boot, iterations, cfg) {
  call_prevalence_safe(
    index = index_date_chr,
    num_years_to_estimate = cfg$num_years_to_estimate,
    data = dat,
    inc_formula  = entrydate ~ sex,
    surv_formula = Surv(time, status) ~ age + sex,
    dist = "weibull",
    population_size = 1e6,
    death_column = "eventdate",
    N_boot = N_boot,
    iterations = iterations
  )
}

# ---- Mérés: K-szor hívjuk a prevalence()-t, és mérjük az összidőt ----
measure_one_setting <- function(N, K, N_boot, iterations, cfg, seed = 1L) {
  dat <- make_registry(N, seed = seed)

  idx_dates <- make_index_dates(K, cfg$index_start, cfg$index_step_days)

  # KRITIKUS FIX:
  # for(d in idx_dates) közben a Date class le tud esni -> előre karakterré alakítjuk.
  idx_dates_chr <- format(idx_dates, "%Y-%m-%d")

  if (length(idx_dates_chr) != K || anyNA(idx_dates_chr)) {
    stop("Index dátum generálás hibás: idx_dates_chr NA-t tartalmaz vagy rossz hossz.")
  }

  elapsed <- numeric(cfg$reps)

  for (r in seq_len(cfg$reps)) {
    gc()
    t <- system.time({
      for (d_chr in idx_dates_chr) {
        invisible(run_rprev_once(dat, d_chr, N_boot, iterations, cfg))
      }
    })
    elapsed[r] <- unname(t[["elapsed"]])
  }

  data.frame(
    N = N,
    K = K,
    N_boot = N_boot,
    reps = cfg$reps,
    iterations = iterations,
    elapsed_sec_mean = mean(elapsed)
  )
}

# ---- Grid futtatás ----
results <- do.call(
  rbind,
  lapply(CFG$N_values, function(N) {
    do.call(rbind, lapply(CFG$K_values, function(K) {
      do.call(rbind, lapply(CFG$N_boot_values, function(N_boot) {
        do.call(rbind, lapply(CFG$iterations_values, function(iterations) {
          message(sprintf("Run: N=%g, K=%d, N_boot=%d, iterations=%d", N, K, N_boot, iterations))
          measure_one_setting(
            N = N,
            K = K,
            N_boot = N_boot,
            iterations = iterations,
            cfg = CFG,
            seed = 1000 + as.integer(N) + as.integer(K) + as.integer(N_boot) + as.integer(iterations)
          )
        }))
      }))
    }))
  })
)

results <- results[order(results$N, results$K, results$N_boot, results$iterations), ]
print(results)

# write.csv(results, "rprev_runtime_pilot.csv", row.names = FALSE)

Run: N=1000, K=1, N_boot=100, iterations=1000

Run: N=1000, K=1, N_boot=500, iterations=1000

Run: N=1000, K=1, N_boot=1000, iterations=1000

Run: N=1000, K=10, N_boot=100, iterations=1000

Run: N=1000, K=10, N_boot=500, iterations=1000

Run: N=1000, K=10, N_boot=1000, iterations=1000

Run: N=1000, K=30, N_boot=100, iterations=1000

Run: N=1000, K=30, N_boot=500, iterations=1000

Run: N=1000, K=30, N_boot=1000, iterations=1000

Run: N=10000, K=1, N_boot=100, iterations=1000

Run: N=10000, K=1, N_boot=500, iterations=1000

Run: N=10000, K=1, N_boot=1000, iterations=1000

Run: N=10000, K=10, N_boot=100, iterations=1000

Run: N=10000, K=10, N_boot=500, iterations=1000

Run: N=10000, K=10, N_boot=1000, iterations=1000

Run: N=10000, K=30, N_boot=100, iterations=1000

Run: N=10000, K=30, N_boot=500, iterations=1000

Run: N=10000, K=30, N_boot=1000, iterations=1000



       N  K N_boot reps iterations elapsed_sec_mean
1   1000  1    100    3       1000         1.273333
2   1000  1    500    3       1000         6.423333
3   1000  1   1000    3       1000        16.783333
4   1000 10    100    3       1000        18.083333
5   1000 10    500    3       1000       107.266667
6   1000 10   1000    3       1000       145.260000
7   1000 30    100    3       1000        39.310000
8   1000 30    500    3       1000       226.950000
9   1000 30   1000    3       1000       437.830000
10 10000  1    100    3       1000         6.710000
11 10000  1    500    3       1000        35.670000
12 10000  1   1000    3       1000        66.100000
13 10000 10    100    3       1000        68.740000
14 10000 10    500    3       1000       344.360000
15 10000 10   1000    3       1000       711.616667
16 10000 30    100    3       1000       195.150000
17 10000 30    500    3       1000      1014.196667
18 10000 30   1000    3       1000      2125.900000
